# Lab 08a · What a vision model does when it cannot read the page

[Open in Colab](https://colab.research.google.com/github/AIGuruOne/OQ-labs/blob/main/notebooks/08a_vision_basics.ipynb)

**Day 3 · S16 · START HERE** · Budget: 15 min · **One hosted API key. No GPU, no Ollama, no local model, nothing to install beyond one package.**

You do not need to know any computer vision to do this lab. There are no image
measurements, no scores, no thresholds. There is one work order, photographed three
ways, and one question asked of a model.

**The one idea:** a vision model that cannot read a field does not tell you. It
returns a value of the right shape, and nothing downstream can tell the difference.

**A correct result looks like:** on the clean scan the model reads all six fields
correctly. On the poor photograph it gets five of six **wrong** while reporting no
error at all — and the obvious fix, telling it to say so when unsure, **does not
work**. Only an explicit instruction to assume it cannot read makes it back off.

> All data in this lab is synthetic. No real OQ material anywhere.

## 1. Setup

One package, and only if you are on Colab. `openai` is the vendor SDK; everything
else is preinstalled.

In [ ]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai==3.0.0"], check=True)

import base64, io, json, os, re
from PIL import Image, ImageDraw, ImageFilter, ImageFont
from openai import OpenAI

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

MODEL = "gpt-4.1-mini"
client = OpenAI(timeout=120, default_headers={"Accept-Encoding": "gzip"})
print(f"runtime: {'Colab' if IN_COLAB else 'local'} | model: {MODEL} | key: {'set' if os.environ.get('OPENAI_API_KEY') else 'MISSING'}")

## 2. One work order

A maintenance work order, drawn here so we know exactly what is on it. Those six
values are the answer key for the rest of the notebook.

In [ ]:
TRUTH = {"work_order": "WO-118305", "equipment": "P-1204A", "site": "MRB",
         "permit": "PTW-48213", "discipline": "Mechanical", "signoff_date": "2026-09-14"}
LABELS = [("Work order","work_order"), ("Equipment","equipment"), ("Site","site"),
          ("Permit","permit"), ("Discipline","discipline"), ("Sign-off date","signoff_date")]

def font(sz):
    for n in ("DejaVuSans.ttf","Arial.ttf","Helvetica.ttc"):
        try: return ImageFont.truetype(n, sz)
        except Exception: pass
    return ImageFont.load_default()

def work_order():
    im = Image.new("RGB",(840,560),"white"); d = ImageDraw.Draw(im)
    d.text((40,28), "MAINTENANCE WORK ORDER", fill="black", font=font(26))
    d.line((40,66,800,66), fill="black", width=3)
    y = 96
    for label, key in LABELS:
        d.text((44,y), label, fill="#333", font=font(19))
        d.line((250,y+26,560,y+26), fill="#999", width=1)
        d.text((256,y), TRUTH[key], fill="black", font=font(19)); y += 46
    d.text((44,y+16), "READINGS", fill="#333", font=font(19)); y += 48
    for x,c in zip([44,240,420,600], ["Point","Before","After","Unit"]):
        d.text((x,y), c, fill="#333", font=font(16))
    y += 24
    for p,b,a in [("TP-1","4.6","2.1"), ("TP-2","5.2","2.4")]:
        d.line((40,y-4,800,y-4), fill="#ccc", width=1)
        for x,v in zip([44,240,420,600], [p,b,a,"mm/s"]): d.text((x,y), v, fill="black", font=font(16))
        y += 28
    return im

FORM = work_order()
print("six fields, and we know all six:")
for label, key in LABELS: print(f"   {label:14s} {TRUTH[key]}")
display(FORM)

## 3. Photograph it three ways

Nobody sends a flatbed scan from the field. They send a phone photograph — sometimes
a good one, sometimes taken in a dim plant room from too far back.

Three versions of the *same* form. Only the photograph changes; the values on it
never do.

In [ ]:
def photograph(blur, scale, quality):
    im = FORM.convert("L")
    im = im.resize((int(im.width*scale), int(im.height*scale)))
    if blur: im = im.filter(ImageFilter.GaussianBlur(blur))
    buf = io.BytesIO(); im.save(buf, format="JPEG", quality=quality)
    return buf.getvalue()

SHOTS = {
    "clean scan":   photograph(0.0, 1.00, 92),
    "decent photo": photograph(1.0, 0.60, 60),
    "poor photo":   photograph(1.6, 0.45, 40),
}
for name, data in SHOTS.items():
    print(f"{name:14s} {len(data)/1024:6.1f} KB")
display(Image.open(io.BytesIO(SHOTS["poor photo"])))
print("^ the poor photograph. Squint at it: can YOU read the permit number?")

## 4. Ask the model to read it

One prompt. Six keys. Nothing clever.

In [ ]:
KEYS = ", ".join(k for _, k in LABELS)
NAIVE = f"Read this maintenance work order and return ONLY JSON with exactly these keys: {KEYS}."

def read_form(png, prompt):
    b64 = base64.b64encode(png).decode()
    r = client.chat.completions.create(model=MODEL, temperature=0, max_tokens=400,
        messages=[{"role":"user","content":[
            {"type":"text","text":prompt},
            {"type":"image_url","image_url":{"url":f"data:image/jpeg;base64,{b64}"}}]}])
    text = r.choices[0].message.content
    return json.loads(re.search(r"\{.*\}", text, re.S).group())

def compare(got, title):
    wrong = refused = 0
    print(f"--- {title}")
    print(f"    {'field':14s}{'on the form':16s}{'model read':16s}")
    for label, key in LABELS:
        value = str(got.get(key, "")).strip()
        if "NOT_LEGIBLE" in value.upper(): mark, refused = "refused", refused + 1
        elif value == TRUTH[key]:          mark = "ok"
        else:                              mark, wrong = "WRONG", wrong + 1
        print(f"    {label:14s}{TRUTH[key]:16s}{value:16s}{mark}")
    print(f"    => {6-wrong-refused} correct, {wrong} wrong, {refused} refused\n")
    return wrong, refused

compare(read_form(SHOTS["clean scan"], NAIVE), "CLEAN SCAN, plain prompt")

All six correct. This is the part that builds the false confidence: on a good image a
current vision model reads a form like this perfectly, every time.

Now the same prompt on the photograph you could barely read yourself.

In [ ]:
wrong_naive, _ = compare(read_form(SHOTS["poor photo"], NAIVE), "POOR PHOTO, plain prompt")

## 5. Stop and look at what just happened

Not "the model failed". Look at *how* it failed.

- The JSON is **valid**.
- Every value is the **right shape**: a work order that looks like a work order
  number, a tag that looks like an equipment tag, a date that looks like a date.
- `PTW-48213` came back as something like `PTW-48715` — same prefix, same length,
  different number.
- There is **no error, no warning, no confidence score, no empty field**.

A system that received this would accept all six values. A person reviewing the
output, without the photograph next to it, would have no reason to doubt any of them.
The wrong permit number is the one that should worry you: permits control who is
allowed to do dangerous work.

## 6. The fix everybody writes first

The obvious answer is to tell it to say so when it is not sure. Let us try exactly
that.

In [ ]:
SOFT = (f"Read this maintenance work order and return ONLY JSON with exactly these keys: {KEYS}.\n"
        "Only report a value you can actually READ in the image, character by character.\n"
        'If a value is blurred, cut off, or you are guessing any character, write "NOT_LEGIBLE".\n'
        "A wrong value is far worse than NOT_LEGIBLE. Do not infer a value from the shape of the field.")

wrong_soft, refused_soft = compare(read_form(SHOTS["poor photo"], SOFT), "POOR PHOTO, 'say if unsure'")
print("Did asking politely help?", "yes" if refused_soft else "NO - it changed nothing")

**It changes nothing.** The instruction reads as careful, it would pass a code review,
and the model still returns the same confident wrong values.

This is the most useful thing in the lab. A polite instruction is not a control.

## 7. What does work

Be explicit about the thing you actually distrust: assume the image is unreadable,
make refusal the default, and name identifiers as the highest-risk fields.

In [ ]:
STRICT = ("You are transcribing a photograph of a maintenance work order.\n\n"
          f"Return ONLY JSON with exactly these keys: {KEYS}.\n\n"
          "RULES:\n"
          "1. This photograph is low resolution. Assume you CANNOT read most of it.\n"
          "2. For each field ask: can I make out every single character with certainty?\n"
          '3. If the answer is no for ANY character, the value is exactly "NOT_LEGIBLE".\n'
          "4. Identifiers (work order, equipment tag, permit, dates) are the ones to be\n"
          "   strictest about: a plausible-looking wrong identifier is the worst output.\n"
          '5. Never reconstruct a value from its expected format. "It looks like a permit\n'
          '   number" is not reading it. When in doubt, "NOT_LEGIBLE".')

compare(read_form(SHOTS["poor photo"], STRICT), "POOR PHOTO, 'assume you cannot read'")

### And check it has not just become useless

A model that refuses everything is no better than one that guesses everything. Run
the strict prompt against the clean scan: it should still read all six.

In [ ]:
compare(read_form(SHOTS["clean scan"], STRICT), "CLEAN SCAN, 'assume you cannot read'")

## 8. What to take away

**A vision model does not read a form. It interprets one.** On a good image that
interpretation is right. On a bad image it is still confident, still well-formed, and
wrong — and nothing in the output distinguishes the two.

**The failure has no error attached to it.** No exception, no empty field, no
confidence score. Everything a normal pipeline checks for would pass. That is what
makes it dangerous against a system of record.

**"Tell it to say when it is unsure" is not a control.** You watched it do nothing.
What worked was inverting the default: assume unreadable, refuse unless certain, and
name the fields where a wrong answer costs the most.

**Refusing has a price.** The strict prompt refuses fields it could have read. That
is a trade you choose deliberately: a `NOT_LEGIBLE` is a person re-photographing a
page; a wrong permit number is a person working under the wrong permit.

### Where this goes next

| | |
|---|---|
| **Lab 09** | the same failure measured over ten forms, with a self-hosted model alongside the vendor one |
| **Lab 10** | what happens when these extractions get indexed and retrieved as if they were true |
| **Day 4** | the approval interrupt: which of these values may a machine act on without a person |
| **Day 5** | this is a row in the governance pack — what the model may read, who checks it, what happens when it is wrong |

In [ ]:
print("08a VISION BASICS OK")
print(f"  clean scan  : read correctly")
print(f"  poor photo  : {wrong_naive} of 6 wrong with a plain prompt, and {wrong_soft} of 6 still wrong")
print(f"                after asking it to flag uncertainty")
print( "  the fix that worked was telling it to assume it could not read at all")